In [1]:
import sys
import os

import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import warnings

warnings.filterwarnings("ignore")

In [2]:
import pandas as pd

In [3]:
dist_out_dir = '/home/workspace/spatial_mouse_lung_outputs/downstream_analysis/distance'
dist_out_dir_cellchat = '/home/workspace/spatial_mouse_lung_outputs/downstream_analysis/distance/cellchat_zones_il21'

if not os.path.exists (dist_out_dir_cellchat):
    os.makedirs(dist_out_dir_cellchat)

plot_out_dir = os.path.join(dist_out_dir_cellchat, 'plots')
if not os.path.exists (plot_out_dir):
    os.makedirs(plot_out_dir)

In [4]:
adata = sc.read_h5ad(os.path.join(dist_out_dir,'adata_distance_zones_structure_tls_dist_categories.h5ad'))


In [5]:
adata.obs['label_fine'].unique().tolist()

['Col13a1+ fibroblast',
 'Alv Mf',
 'Cap',
 'Vein',
 'AT2',
 'Mono',
 'Th0',
 'Pericyte 2',
 'Cap-a',
 'Neut',
 'Pericyte 1',
 'Club',
 'Ciliated',
 'Art',
 'AT1',
 'CD4 naive',
 'B cell',
 'Th17',
 'Int Mf',
 'CD8 naive',
 'SMC',
 'gd T cell',
 'Plasmablast',
 'Th2',
 'Lymph',
 'Ccr7- cDC2',
 'NK cell',
 'cDC1',
 'CD4 trans',
 'Ccr7+ cDC2',
 'Th1',
 'CD8 act',
 'Mesothelial',
 'Treg',
 'Myofibroblast',
 'Col14a1+ fibroblast',
 'ILC2']

In [7]:
adata.obs.columns

Index(['cell_id', 'x_centroid', 'y_centroid', 'transcript_counts',
       'control_probe_counts', 'genomic_control_counts',
       'control_codeword_counts', 'unassigned_codeword_counts',
       'deprecated_codeword_counts', 'total_counts', 'cell_area',
       'nucleus_area', 'nucleus_count', 'segmentation_method', 'sample',
       'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts',
       'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes',
       'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'n_counts',
       'sample_label', 'orig.ident', 'nCount_RNA', 'nFeature_RNA',
       'percent.mito', 'Barcode', 'Age', 'Oxygen', 'percent.mt', 'S.Score',
       'G2M.Score', 'Phase', 'Sample', 'RNA_snn_res.0.2', 'seurat_clusters',
       'RNA_snn_res.0.15', 'nCount_SCT', 'nFeature_SCT',
       'integrated_snn_res.0.2', 'integrated_snn_res.0.1', 'cluster_high_res',
       'CellType', 'leiden_res1', 'CellType_consolidated', '_scvi_batch',
       '_scvi_label

### Categorize CD4 by zone

In [8]:
# focus only on TLS, adventitia, parenchyma
adata = adata[adata.obs['zone_consol'].isin(['TLS', 'adventitia', 'parenchyma']), :]
# set labels
adata.obs['zone_consol'] = adata.obs['zone_consol'].values.tolist()
adata.obs['label_fine'] = adata.obs['label_fine'].values.tolist()

In [9]:
adata.shape

(514929, 480)

In [10]:
#wherever cell_type_1 is 'T Cells', add the classification to the cell_type_1 column
# Create a mask for CD4 act
cd4_mask = adata.obs['label_fine'].isin(['Th0', 'Th1', 'Th17', 'Th2', 'Treg', 'CD4 trans'])

# make single activated T cell label per region 
adata.obs.loc[cd4_mask, 'label_fine'] = (
    'CD4 act (' + adata.obs.loc[cd4_mask, 'zone_consol'] + ')'
)

# Check the updated cell types
print("Updated T cell categories:")
print(adata.obs.loc[cd4_mask, 'label_fine'].value_counts())

# adata.obs['label_medium']


Updated T cell categories:
label_fine
CD4 act (parenchyma)    6313
CD4 act (adventitia)    2409
CD4 act (TLS)           1479
Name: count, dtype: int64


In [13]:
# Check the updated cell types
adata_d3 = adata[adata.obs['sample_label']=='HDM_day3', :]
adata_d30 = adata[adata.obs['sample_label']=='HDM_day30', :]
print("Updated T cell categories, day 3:")
cd4_mask = adata_d3.obs['label_fine'].isin(['CD4 act (parenchyma)', 'CD4 act (adventitia)', 'CD4 act (TLS)'])
print(adata_d3.obs.loc[cd4_mask, 'label_fine'].value_counts())

print("Updated T cell categories, day 30:")
cd4_mask = adata_d30.obs['label_fine'].isin(['CD4 act (parenchyma)', 'CD4 act (adventitia)', 'CD4 act (TLS)'])
print(adata_d30.obs.loc[cd4_mask, 'label_fine'].value_counts())

Updated T cell categories, day 3:
label_fine
CD4 act (parenchyma)    5918
CD4 act (adventitia)    1384
CD4 act (TLS)           1200
Name: count, dtype: int64
Updated T cell categories, day 30:
label_fine
CD4 act (adventitia)    1025
CD4 act (parenchyma)     395
CD4 act (TLS)            279
Name: count, dtype: int64


In [14]:
dist_out_dir_cellchat

'/home/workspace/spatial_mouse_lung_outputs/downstream_analysis/distance/cellchat_zones_il21'

In [15]:
adata.write_h5ad(os.path.join(dist_out_dir_cellchat,'adata_cellchat_prepped.h5ad'))